# Predicted restaurant busyness across Manhattan — Saturday 20:00

A single-panel companion to `manhattan_busyness_figure.ipynb`. That notebook
draws three panels (08:00 / 14:00 / 20:00 on a **Friday**); this one draws
**one** map: **20:00 on a Saturday in July**, over the same 300 restaurants
selected the same way.

As in the sibling notebook, predictions come from the **deployed** model —
`busyness_xgboost_pipeline.joblib` via `BusynessModelService.predict`, the same
entry point behind the FastAPI service's `/predict/busyness` — so the figure
shows what the running system returns rather than a reimplementation.

### On "a random Saturday night in July"

The model takes **`(month, weekday, hour)`**, not a calendar date. Saturday
evening in July is therefore `month=7, weekday=5, hour=20`, and *every*
Saturday in July produces an identical prediction. The cell below draws one
July 2025 Saturday with a fixed seed purely so the figure can be captioned
with a concrete date; changing the seed changes the caption, never the numbers.
Nothing downstream reads the date.

**Requirements:** `scikit-learn==1.8.0` (the version the pipeline was fitted
with), `xgboost`, `pandas`, `pyarrow`, `joblib`, `pyshp`, `pyproj`,
`matplotlib`. No network access and no basemap service — the island outline
comes from the TLC taxi-zone shapefile already in
`ml-pipeline/fastapi-app/data/`.

In [ ]:
import random
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapefile                      # pyshp
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon as MplPolygon
from pyproj import Transformer

%matplotlib inline

# --- point this at the repository -------------------------------------------
REPO_ROOT = None
if REPO_ROOT is None:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "ml-pipeline" / "fastapi-app" / "model_service.py").exists():
            REPO_ROOT = candidate
            break
if REPO_ROOT is None:
    raise SystemExit(
        "Set REPO_ROOT to your comp47360-team2 checkout, e.g.\n"
        "    REPO_ROOT = Path(r'C:/Users/you/Documents/GitHub/comp47360-team2')"
    )

FASTAPI_APP = Path(REPO_ROOT) / "ml-pipeline" / "fastapi-app"
OUT_DIR = Path.cwd()
sys.path.insert(0, str(FASTAPI_APP))
print("repo      :", REPO_ROOT)
print("model dir :", FASTAPI_APP)

In [ ]:
# --- prediction setting ------------------------------------------------------
HOUR = 20            # 20:00 — the evening peak
WEEKDAY = 5          # Monday=0 ... Saturday=5, matching the service's convention
MONTH = 7            # July, matching the shipped TLC aggregates

# Caption date only. The model never sees it — see the header.
SEED = 47360
_saturdays = [d for d in pd.date_range("2025-07-01", "2025-07-31") if d.weekday() == 5]
FIGURE_DATE = random.Random(SEED).choice(_saturdays)

assert FIGURE_DATE.weekday() == WEEKDAY, "caption date must be a Saturday"
assert FIGURE_DATE.month == MONTH, "caption date must be in July"

print("July 2025 Saturdays :", [d.strftime('%d %b') for d in _saturdays])
print("drawn for caption   :", FIGURE_DATE.strftime('%A %d %B %Y'), f"at {HOUR:02d}:00")
print("model inputs        :", f"month={MONTH}, weekday={WEEKDAY}, hour={HOUR}")

# --- marker selection --------------------------------------------------------
TARGET = 300         # number of restaurants to plot
MIN_SEP = 130.0      # metres; no two markers may sit closer than this

# Manhattan bounding box, used only to reject corrupted geocodes.
LAT_MIN, LAT_MAX = 40.68, 40.88
LON_MIN, LON_MAX = -74.03, -73.90

CLASS_COLOURS = {0: "#1f4e79", 1: "#e8a33d", 2: "#b5232b"}
CLASS_NAMES = {0: "No Wait", 1: "Queue Required", 2: "Severe Queue"}

## 1. Load the deployed model

In [ ]:
from model_service import BusynessModelService, RestaurantFeatures

svc = BusynessModelService()
feat = svc.restaurant_features

uni = feat[
    feat.latitude.between(LAT_MIN, LAT_MAX)
    & feat.longitude.between(LON_MIN, LON_MAX)
].copy()
uni["taxi_zone_id"] = uni["taxi_zone_id"].astype(int)

print(f"modelling universe : {len(feat)} venues")
print(f"usable geocodes    : {len(uni)}  ({len(feat) - len(uni)} dropped)")
print(f"taxi zones         : {uni.taxi_zone_id.nunique()}")

## 2. Choose 300 restaurants spread across the island

Identical to the sibling notebook, so the two figures plot the **same venues**
and can be compared marker for marker.

- Each taxi zone gets a quota proportional to how many restaurants it holds
  (floor of 1, so no zone disappears).
- Within a zone, each pick is the candidate farthest from everything already
  chosen anywhere on the island, rejected outright if it lands within
  `MIN_SEP` of an existing marker.
- Quota a dense zone cannot fill at that spacing is handed to zones with room.

The procedure is deterministic — no RNG — so it reproduces the sibling
notebook's selection exactly.

In [ ]:
def to_metres(lat, lon):
    """Local equirectangular projection — accurate enough over one island."""
    lat0 = np.deg2rad(40.78)
    return (
        np.deg2rad(lon) * np.cos(lat0) * 6371000.0,
        np.deg2rad(lat) * 6371000.0,
    )


uni["mx"], uni["my"] = to_metres(uni.latitude.values, uni.longitude.values)

# proportional quota per zone
counts = uni.groupby("taxi_zone_id").size()
raw = counts / counts.sum() * TARGET
quota = np.floor(raw).astype(int).clip(lower=1)

while quota.sum() < TARGET:
    for z in (raw - quota).sort_values(ascending=False).index:
        if quota[z] < counts[z]:
            quota[z] += 1
            break
    else:
        break
while quota.sum() > TARGET:
    for z in (raw - quota).sort_values().index:
        if quota[z] > 1:
            quota[z] -= 1
            break


def sweep(candidates, selected_pts, want):
    """Greedy farthest-point picks honouring the global MIN_SEP floor."""

    P = candidates[["mx", "my"]].values
    d = (
        np.sqrt(((P[:, None] - selected_pts[None]) ** 2).sum(2)).min(1)
        if len(selected_pts)
        else np.full(len(candidates), np.inf)
    )
    out = []
    for _ in range(want):
        j = int(np.argmax(d))
        if d[j] < MIN_SEP:
            break
        out.append(candidates.index[j])
        selected_pts = np.vstack([selected_pts, P[j]])
        d = np.minimum(d, np.sqrt(((P - P[j]) ** 2).sum(1)))
    return out, selected_pts


chosen, pts = [], np.empty((0, 2))
for z in counts.sort_values(ascending=False).index:
    got, pts = sweep(uni[uni.taxi_zone_id == z], pts, int(quota[z]))
    chosen += got

# redistribute whatever the dense zones could not fit
for z in counts.sort_values(ascending=False).index:
    if len(chosen) >= TARGET:
        break
    spare = uni[(uni.taxi_zone_id == z) & (~uni.index.isin(chosen))]
    if spare.empty:
        continue
    got, pts = sweep(spare, pts, TARGET - len(chosen))
    chosen += got

picked = uni.loc[chosen].copy()

D = np.sqrt(((pts[:, None] - pts[None]) ** 2).sum(2))
np.fill_diagonal(D, np.inf)
nn = D.min(1)

print(f"selected           : {len(picked)} venues")
print(f"taxi zones covered : {picked.taxi_zone_id.nunique()}")
print(f"nearest neighbour  : min {nn.min():.0f} m   "
      f"median {np.median(nn):.0f} m   max {nn.max():.0f} m")
print(f"bounding box       : lat {picked.latitude.min():.4f}..{picked.latitude.max():.4f}   "
      f"lon {picked.longitude.min():.4f}..{picked.longitude.max():.4f}")

## 3. Predict — Saturday 20:00

One call per venue, straight through the service class. 300 predictions this
time rather than 900, since there is a single hour.

In [ ]:
rows = []
for r in picked.itertuples():
    p = svc.predict(
        hour=HOUR,
        weekday=WEEKDAY,
        month=MONTH,
        restaurant_id=str(r.restaurant_id),
        features=RestaurantFeatures(
            latitude=float(r.latitude),
            longitude=float(r.longitude),
            capacity=None if pd.isna(r.capacity) else int(r.capacity),
        ),
    )
    rows.append({
        "restaurant_id": r.restaurant_id,
        "name": r.original_name,
        "latitude": float(r.latitude),
        "longitude": float(r.longitude),
        "taxi_zone_id": int(p.taxi_zone_id) if p.taxi_zone_id else r.taxi_zone_id,
        "taxi_zone_name": r.taxi_zone_name,
        "weekday": WEEKDAY,
        "hour": HOUR,
        "busyness_level": p.busyness_level,
        "busyness_label": p.busyness_label,
        "busyness_score": p.busyness_score,
        "confidence": p.confidence,
        "taxi_dropoffs_1h": p.taxi_dropoffs_1h,
        "feature_source": p.feature_source,
    })

pred = pd.DataFrame(rows)

print("feature provenance :", pred.feature_source.value_counts().to_dict())
print("distinct scores    :", int(pred.busyness_score.nunique()))
print()
print(f"mean score         : {pred.busyness_score.mean():.4f}")
print(f"median score       : {pred.busyness_score.median():.4f}")
print(f"mean confidence    : {pred.confidence.mean():.4f}")
print()
summary = (
    pred.busyness_label.value_counts()
    .rename_axis("class")
    .to_frame("venues")
    .assign(share=lambda d: (100 * d.venues / len(pred)).round(1))
)
display(summary)

## 4. Basemap from the TLC taxi-zone shapefile

`taxi_zones.zip` ships with the inference container. Reading the Manhattan
polygons out of it gives a real island outline with no tile service, no API key
and no network call. Coordinates are NY State Plane (EPSG:2263), so they need
reprojecting to WGS84.

In [ ]:
CENTRAL_PARK_ZONE = 43

to_wgs = Transformer.from_crs("EPSG:2263", "EPSG:4326", always_xy=True)
sf = shapefile.Reader(str(FASTAPI_APP / "data" / "taxi_zones.zip"))
fields = [f[0] for f in sf.fields[1:]]
bi, zi = fields.index("borough"), fields.index("LocationID")

island, park = [], []
for shp, rec in zip(sf.shapes(), sf.records()):
    if rec[bi] != "Manhattan":
        continue
    bounds = list(shp.parts) + [len(shp.points)]
    for a, b in zip(bounds[:-1], bounds[1:]):
        ring = np.array(shp.points[a:b])
        if len(ring) < 3:
            continue
        lon, lat = to_wgs.transform(ring[:, 0], ring[:, 1])
        (park if int(rec[zi]) == CENTRAL_PARK_ZONE else island).append(
            np.column_stack([lon, lat])
        )

print(f"{len(island)} island rings, {len(park)} Central Park rings")

# Neighbourhood labels sit in the rivers, so they never fall under a marker.
# (text, lat, lon, horizontal alignment)
ANCHORS = [
    ("Inwood",             40.869, -73.948, "right"),
    ("Washington Hts.",    40.840, -73.966, "right"),
    ("Harlem",             40.812, -73.920, "left"),
    ("Upper West Side",    40.790, -74.002, "right"),
    ("Upper East Side",    40.772, -73.930, "left"),
    ("Midtown",            40.752, -73.947, "left"),
    ("Chelsea",            40.744, -74.020, "right"),
    ("Greenwich Village",  40.733, -73.967, "left"),
    ("Lower East Side",    40.714, -73.960, "left"),
    ("Financial District", 40.703, -73.985, "left"),
]

## 5. Draw the figure

One marker per restaurant, opaque, coloured by predicted class — deliberately
**not** a kernel-density blur, so a reader can count venues and see how the
Saturday evening peak distributes across the whole island rather than
concentrating in one hotspot.

In [ ]:
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.edgecolor": "#cccccc",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})

fig, ax = plt.subplots(figsize=(7.6, 9.9))

ax.set_facecolor("#eef3f7")                          # water

for ring in island:
    ax.add_patch(MplPolygon(ring, closed=True, facecolor="#f4f2ee",
                            edgecolor="#dcd8d2", linewidth=0.4, zorder=1))
for ring in park:
    ax.add_patch(MplPolygon(ring, closed=True, facecolor="#dde8d6",
                            edgecolor="#cbd9c2", linewidth=0.4, zorder=2))

for level in (0, 1, 2):
    s = pred[pred.busyness_level == level]
    ax.scatter(s.longitude, s.latitude, s=34, c=CLASS_COLOURS[level],
               edgecolors="white", linewidths=0.6, zorder=4)

for text, la, lo, ha in ANCHORS:
    ax.text(lo, la, text, fontsize=7.5, color="#9a958e",
            ha=ha, va="center", style="italic", zorder=3)

ax.set_xlim(-74.028, -73.906)
ax.set_ylim(40.694, 40.884)
ax.set_aspect(1 / np.cos(np.deg2rad(40.78)))         # true shape at this latitude
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_linewidth(0.6)

counts_by_class = {lvl: int((pred.busyness_level == lvl).sum()) for lvl in (0, 1, 2)}
handles = [
    Line2D([], [], marker="o", linestyle="none", markersize=8,
           markerfacecolor=CLASS_COLOURS[lvl], markeredgecolor="white",
           label=f"{CLASS_NAMES[lvl]}  —  {counts_by_class[lvl]} venues "
                 f"({100 * counts_by_class[lvl] / len(pred):.0f}%)")
    for lvl in (0, 1, 2)
]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False,
           bbox_to_anchor=(0.5, 0.052), fontsize=8.5)

fig.text(0.5, 0.026,
         f"One marker per restaurant · {len(pred)} venues across all "
         f"{picked.taxi_zone_id.nunique()} Manhattan taxi zones, minimum "
         f"{MIN_SEP:.0f} m apart\n"
         f"Model inputs: month={MONTH}, weekday={WEEKDAY} (Saturday), "
         f"hour={HOUR} · mean predicted score {pred.busyness_score.mean():.3f}",
         ha="center", fontsize=7.5, color="#666666", linespacing=1.5)

fig.suptitle("Predicted restaurant busyness across Manhattan",
             fontsize=15, fontweight="bold", y=0.972)
ax.set_title(f"{FIGURE_DATE.strftime('%A %d %B %Y')} · {HOUR:02d}:00",
             fontsize=11, color="#444444", pad=10)

fig.subplots_adjust(left=0.03, right=0.97, top=0.925, bottom=0.10)

fig.savefig(OUT_DIR / "fig_manhattan_busyness_saturday_2000.png", dpi=300)
plt.show()
print("saved fig_manhattan_busyness_saturday_2000.png to", OUT_DIR)

## 6. Numbers for the write-up

District rollup, so any figure quoted in the paper comes from the same run that
drew the panel.

In [ ]:
DISTRICTS = {
    "Lower Manhattan": [13, 87, 88, 209, 231, 261, 45, 232, 144, 211, 125],
    "Village & LES": [79, 4, 113, 114, 148, 249, 158],
    "Chelsea, Flatiron & Union Sq": [68, 90, 107, 234, 246, 224, 137],
    "Midtown": [48, 50, 100, 161, 162, 163, 164, 170, 186, 229, 230, 233],
    "Upper West Side": [24, 142, 143, 151, 238, 239, 166],
    "Upper East Side": [140, 141, 236, 237, 262, 263, 202],
    "Harlem": [41, 42, 74, 75, 152],
    "Upper Manhattan": [116, 127, 243, 244],
}
lookup = {z: name for name, zones in DISTRICTS.items() for z in zones}
pred["district"] = pred.taxi_zone_id.map(lookup)

rollup = (
    pred.groupby("district")
    .agg(
        venues=("busyness_score", "size"),
        mean_score=("busyness_score", "mean"),
        pct_severe=("busyness_level", lambda s: 100 * (s == 2).mean()),
        pct_no_wait=("busyness_level", lambda s: 100 * (s == 0).mean()),
        mean_confidence=("confidence", "mean"),
    )
    .round({"mean_score": 3, "pct_severe": 1, "pct_no_wait": 1, "mean_confidence": 3})
    .sort_values("mean_score", ascending=False)
)
print(f"Saturday {HOUR:02d}:00 — by district, ordered by mean predicted score")
display(rollup)

busiest = rollup.index[0]
quietest = rollup.index[-1]
print(f"busiest district  : {busiest} "
      f"(mean {rollup.loc[busiest, 'mean_score']:.3f}, "
      f"{rollup.loc[busiest, 'pct_severe']:.0f}% Severe Queue)")
print(f"quietest district : {quietest} "
      f"(mean {rollup.loc[quietest, 'mean_score']:.3f}, "
      f"{rollup.loc[quietest, 'pct_no_wait']:.0f}% No Wait)")

pred.to_csv(OUT_DIR / "manhattan_busyness_predictions_saturday_2000.csv", index=False)
print("wrote manhattan_busyness_predictions_saturday_2000.csv")

## 7. Optional — how Saturday 20:00 differs from Friday 20:00

Only meaningful if you have already run `manhattan_busyness_figure.ipynb` in
this directory, since it reads that notebook's CSV. Skips itself otherwise.
The two runs share the same 300 venues, so the comparison is paired.

In [ ]:
friday_csv = OUT_DIR / "manhattan_busyness_predictions.csv"

if not friday_csv.exists():
    print(f"{friday_csv.name} not found — run manhattan_busyness_figure.ipynb "
          "first if you want this comparison. Skipping.")
else:
    fri = pd.read_csv(friday_csv)
    fri20 = fri[fri.hour == 20][["restaurant_id", "busyness_level", "busyness_score"]]
    fri20 = fri20.rename(columns={"busyness_level": "fri_level",
                                  "busyness_score": "fri_score"})
    sat20 = pred[["restaurant_id", "busyness_level", "busyness_score"]].rename(
        columns={"busyness_level": "sat_level", "busyness_score": "sat_score"})

    paired = fri20.merge(sat20, on="restaurant_id", how="inner")
    print(f"venues matched across both runs : {len(paired)} of {len(pred)}")

    if len(paired):
        print(f"mean score  Friday 20:00 : {paired.fri_score.mean():.4f}")
        print(f"mean score  Saturday 20:00: {paired.sat_score.mean():.4f}")
        print(f"mean delta  (Sat - Fri)   : {(paired.sat_score - paired.fri_score).mean():+.4f}")
        print()
        moved_up = int((paired.sat_level > paired.fri_level).sum())
        moved_dn = int((paired.sat_level < paired.fri_level).sum())
        print(f"venues in a busier class on Saturday : {moved_up}")
        print(f"venues in a quieter class on Saturday: {moved_dn}")
        print(f"unchanged                            : {len(paired) - moved_up - moved_dn}")
        print()
        print("class transition, Friday 20:00 -> Saturday 20:00")
        display(pd.crosstab(
            paired.fri_level.map(CLASS_NAMES),
            paired.sat_level.map(CLASS_NAMES),
        ))

## Caveats

- The model's label is the **Google Places popular-times index**, a published
  activity proxy — not observed occupancy or queue length. The panel shows
  predicted published activity.
- Held-out accuracy is 62.7% with QWK 0.599 on venues never seen in training.
  The model leans toward *overstating* waits (36.3% of its middle-class errors
  go up, 17.2% down), so read the red as an upper bound.
- Zone-level taxi demand alone explains almost none of the between-zone
  variation (*r* = 0.102, *R²* = 0.010). The spatial structure here comes from
  venue composition, not from the taxi feature.
- **The date in the title is a caption, not an input.** The model consumes
  `(month, weekday, hour)` only, so this panel represents *any* Saturday
  20:00 in July equally. It is not a forecast for one specific evening, and
  nothing here responds to weather, events or transit disruption on that date.
- These 300 venues are **not** the contents of the seeded application database,
  which is confined to a ~350 m disc around Times Square.
- **Environment note.** The pipeline was fitted under `scikit-learn==1.8.0`.
  Loading it under a different minor version raises
  `InconsistentVersionWarning` and is not guaranteed to reproduce these
  predictions — pin 1.8.0 before quoting any number from this notebook.